# CineForge Colab Video Server — Wan 2.1 T2V 1.3B (realistic, free)  **FIXED v2**

This notebook hosts a **realistic text-to-video model** (Wan 2.1 1.3B, open weights, free) on a free
Colab GPU (T4) and exposes it as a small HTTP API that the CineForge `colab` backend on your laptop
can call.

## &nbsp;

## ⚠️ **STEP 0: SELECT GPU FIRST — DO THIS BEFORE RUNNING ANY CELLS**

Top menu → **Runtime → Change runtime type** →
- Hardware accelerator: **GPU**
- GPU type: **T4** (free tier)
- Runtime shape: **Standard** (High RAM not required but helps — optional)

Then **Save**.  You should see a green tick next to "Connected" in the top-right.

---

## How to use
1. Run **cells 1 → 2** in order.  After cell 2 you **must** click the **RESTART RUNTIME** button that appears.
2. After restart, start again from **cell 3** (the env check cell).
3. Run all remaining cells (3 → 4 → 5 → 6 → 7).  The last cell prints a **PUBLIC URL**.
4. Paste that URL into your laptop's `.env` as `COLAB_BASE_URL=<that URL>` (no trailing slash).
5. On your laptop, generate with CineForge using `--backend colab`.

First generation takes ~3–5 minutes (download + warm-up); after that each 5 s clip takes
roughly 2–4 minutes on a T4 at 480p.  Keep this browser tab open; the URL dies when Colab
disconnects (just re-run cells 5–7 and update `.env`).

In [ ]:
# ==========================================================================
#  CELL 1 — Install dependencies.  VERSIONS ARE PINNED EXPLICITLY.
#  Do not change them unless you know exactly what broke.
# ==========================================================================

# Fix CUDA lazy-loading so bitsandbytes kernel lookup works on T4.
import os
os.environ["CUDA_MODULE_LOADING"] = "EAGER"

import sys
print("Python:", sys.version.split()[0])

# ---- Version pins (all of these matter). ---------------------------------
# transformers 4.53: last version before the "caching allocator warmup" that
#   pre-allocates a full fp16 copy on the GPU *before* 8-bit quantization,
#   guaranteeing OOM on a 16 GB T4.
# diffusers 0.31.0: first WanPipeline tag that is API-stable with transformers 4.53.
#   0.32+ changed from_pretrained() kwargs and breaks our helper.
# bitsandbytes 0.43.3: LAST version whose CUDA kernels actually WORK on the
#   free T4.  0.44.x ships broken LayerNorm kernels and crashes with
#   "LayerNormKernel_cublas not implemented".
# accelerate 1.1.x: compatible with both transformers 4.53 + diffusers 0.31.
# sentencepiece: mandatory for UMT5Tokenizer (Wan's text encoder).
# protobuf<5:  UMT5Tokenizer's internals crash on protobuf 5 (Colab default).
# imageio-ffmpeg: required by diffusers.export_to_video() otherwise it raises.
# --------------------------------------------------------------------------

!pip install -q --no-cache-dir \
    "transformers==4.53.0" \
    "diffusers==0.31.0" \
    "accelerate==1.1.1" \
    "bitsandbytes==0.43.3" \
    "sentencepiece==0.2.0" \
    "protobuf==4.25.5" \
    "safetensors>=0.4.5" \
    "imageio-ffmpeg>=0.5.1" \
    "fastapi==0.115.0" \
    "uvicorn==0.30.6" \
    "nest-asyncio==1.6.0" \
    "pydantic==2.9.2" \
    "psutil" \
    2>&1 | tail -25

print("\n\n✅ pip install finished.")
print("\n")
print("=" * 72)
print("🛑  NEXT STEP — REQUIRED BEFORE CONTINUING  🛑")
print("=" * 72)
print("Top menu →  Runtime →  Restart session")
print("   (or click the blue RESTART RUNTIME button below if one appeared)")
print("After restarting, START AGAIN FROM CELL 3 (env check) — do NOT re-run cell 1.")
print("If you skip the restart, every subsequent cell will fail with ImportError.")
print("=" * 72)


---

## ⚠️ STOP. DID YOU RESTART THE RUNTIME YET?

If you just ran cell 1 and landed here: **Restart Runtime now**.

*Runtime → Restart session.*

Wait for the notebook to reconnect, then click on cell 3 below and press Shift+Enter.

---


In [ ]:
# ==========================================================================
#  CELL 3 — Environment sanity check.  If any line says FAIL, STOP & read.
# ==========================================================================
import os, sys, importlib
os.environ["CUDA_MODULE_LOADING"] = "EAGER"
# Invalidate any stale cached imports left from before the restart.
importlib.invalidate_caches()

def _ok(msg): print(" ✅", msg)
def _fail(msg): print(" ❌ FAIL:", msg); _HAD_FAIL[0] = True
_HAD_FAIL = [False]

# 3a. packages + versions
import importlib.metadata as md
pins = {
    "transformers": "4.53.0",
    "diffusers":    "0.31.0",
    "accelerate":   "1.1.1",
    "bitsandbytes": "0.43.3",
    "sentencepiece":None,
    "protobuf":     None,
    "fastapi":      None,
    "uvicorn":      None,
    "pydantic":     None,
    "safetensors":  None,
    "imageio_ffmpeg": None,
}
for pkg, want in pins.items():
    try:
        got = md.version(pkg)
        if want and got != want:
            _fail(f"{pkg}: expected {want}, got {got}.  Re-run cell 1, then Restart Runtime.")
        else:
            _ok(f"{pkg} {got}")
    except md.PackageNotFoundError:
        _fail(f"{pkg} is not installed.  Re-run cell 1.")

# 3b. torch + CUDA + GPU
try:
    import torch
    _ok(f"torch {torch.__version__}")
except Exception as e:
    _fail(f"torch import: {e}")
    torch = None

if torch is not None:
    if not torch.cuda.is_available():
        _fail("CUDA not visible.  Runtime → Change runtime type → T4 GPU, then Restart.")
    else:
        gpu = torch.cuda.get_device_name(0)
        vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        _ok(f"GPU: {gpu}  ({vram_gb} GB VRAM)")
        if vram_gb < 14.0:
            print("   ⚠️  You got a <16 GB GPU shape (L4 or partial T4).  May still work but"
                  " keep resolutions ≤832×480 and durations ≤5 s.")
        # Pre-warm CUDA context so bitsandbytes never sees "libcuda.so not found"
        _ = torch.zeros((1,), device="cuda")
        torch.cuda.synchronize()
        _ok("CUDA context warmed up.")

# 3c. RAM (must be ≥10 GB to survive the low_cpu_mem_usage load)
try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / 1e9, 1)
    _ok(f"System RAM: {ram_gb} GB")
    if ram_gb < 10.0:
        print("   ⚠️  Low-RAM shape.  If you see 'session crashed for unknown reason' during"
              " model load, upgrade to High-RAM shape.")
except Exception as e:
    _fail(f"psutil: {e}")

# 3d. critical dynamic imports
for name, module_path, symbols in [
    ("diffusers.WanPipeline", "diffusers", ["WanPipeline", "AutoencoderKLWan"]),
    ("transformers.BitsAndBytesConfig", "transformers",
     ["AutoTokenizer", "UMT5EncoderModel", "BitsAndBytesConfig"]),
]:
    try:
        mod = importlib.import_module(module_path)
        for sym in symbols:
            if not hasattr(mod, sym):
                _fail(f"{sym} missing from {module_path}.  Did you Restart Runtime?")
        _ok(f"imports ok: {module_path} has {symbols}")
    except Exception as e:
        _fail(f"import {name}: {e}.  Usually means you skipped Runtime → Restart after cell 1.")

print()
if _HAD_FAIL[0]:
    raise SystemExit("\n🛑 Environment check FAILED.  Fix the lines above and re-run this cell.")
else:
    print("🎉 All environment checks PASSED.  Safe to continue to cell 4.")


In [ ]:
# ==========================================================================
#  CELL 4 — Download + load Wan 2.1 1.3B onto the T4.
#  First run ≈ 3–6 min (9 GB download).  Subsequent runs < 30 s if cached.
# ==========================================================================
import gc, os, sys
import torch
import psutil
import transformers, diffusers
from diffusers import WanPipeline, AutoencoderKLWan
from transformers import AutoTokenizer, UMT5EncoderModel, BitsAndBytesConfig

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
USE_OLD_CACHE_POLICY = True   # HF cache mirrors the Hub snapshot tree.

def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

# ---- 4a. Text encoder → 8-bit straight onto the GPU ------------------------
# low_cpu_mem_usage=True  → never materializes the fp16 weights in system RAM.
#                           This single flag is what prevents RAM-OOM on free-tier Colab.
# bnb_8bit_compute_dtype  → bnb matmul uses fp16 on T4 (default fp32 compute
#                           would double the activation memory and OOM).
# ----------------------------------------------------------------------------
print("① Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
_free()

print("② Loading 8-bit umT5 text encoder onto GPU (low CPU mem path) ...")
bnb_8bit_cfg = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
    bnb_8bit_use_double_quant=False,
)
text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder",
    quantization_config=bnb_8bit_cfg,
    torch_dtype=torch.float16,
    device_map="cuda",     # bitsandbytes handles placement; 'cuda' avoids auto cpu-offload bugs
    low_cpu_mem_usage=True,
)
_free()
print("   RAM after text encoder :", round(psutil.virtual_memory().used / 1e9, 1), "GB")
print("   VRAM after text encoder:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

# ---- 4b. VAE (fp32 — numerically fragile; tiled to fit) ------------------
print("③ Loading VAE (fp32) ...")
vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID, subfolder="vae",
    torch_dtype=torch.float32,
)
vae.to("cuda")
_free()
print("   VRAM after VAE:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

# ---- 4c. Transformer (fp16) + assemble pipeline ---------------------------
print("④ Loading transformer (fp16) & assembling pipeline ...")
pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    vae=vae,
    torch_dtype=torch.float16,
)
pipe.transformer.to("cuda")
pipe.vae.to("cuda")
pipe.enable_vae_tiling()
_free()
print("   VRAM after transformer:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

print()
print("=" * 72)
print(f"✅ Wan 2.1 1.3B ready on: {torch.cuda.get_device_name(0)}")
print("   VRAM used (GB)  :", round(torch.cuda.memory_allocated() / 1e9, 1))
print("   VRAM total (GB) :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("   RAM used  (GB)  :", round(psutil.virtual_memory().used / 1e9, 1))
print("=" * 72)


In [ ]:
# ==========================================================================
#  CELL 5 — Define the FastAPI HTTP server the CineForge colab backend calls.
# ==========================================================================
import io, os, tempfile, threading, time, random
from typing import Optional
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel, Field
from diffusers.utils import export_to_video
import torch

app = FastAPI(title="CineForge Colab Video Server (Wan 2.1 1.3B)")
GEN_LOCK = threading.Lock()       # one gen at a time on free tier

def _snap(v, lo=64, multiple=16):
    """Snap a pixel dim onto the grid Wan's VAE was trained at."""
    return max(lo, (int(v) // multiple) * multiple)

class GenBody(BaseModel):
    prompt: str
    negative_prompt: str = ""
    width:  int = Field(default=832, ge=64,  le=1280)
    height: int = Field(default=480, ge=64,  le=1280)
    fps:    int = Field(default=16,  ge=5,   le=30)
    duration: float = Field(default=5.0, ge=1.0, le=10.0)
    num_inference_steps: int   = Field(default=25,  ge=4,  le=50)
    guidance_scale:    float = Field(default=5.0, ge=0.0, le=15.0)
    seed: Optional[int] = None      # pydantic v1/v2 compatible (no `int | None`)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "busy": GEN_LOCK.locked(),
        "vram_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    }

@app.post("/generate")
def generate(body: GenBody):
    negative = body.negative_prompt or (
        "cartoon, anime, illustration, painting, drawing, CGI, 3d render, "
        "plastic skin, blurry, low quality, worst quality, jpeg artifacts, deformed, "
        "extra limbs, missing fingers, bad hands, bad teeth"
    )
    seed = body.seed if body.seed is not None else random.randint(0, 2**31 - 1)
    gen = torch.Generator("cuda").manual_seed(seed)
    num_frames = max(9, min(int(body.duration * body.fps), 121))
    w, h = _snap(body.width), _snap(body.height)
    t0 = time.time()

    with GEN_LOCK:
        try:
            out = pipe(
                prompt=body.prompt,
                negative_prompt=negative,
                width=w, height=h,
                num_frames=num_frames,
                num_inference_steps=body.num_inference_steps,
                guidance_scale=body.guidance_scale,
                generator=gen,
                max_sequence_length=512,       # umT5 default (226) truncates long prompts!
            ).frames[0]
        except torch.cuda.OutOfMemoryError as e:
            gc.collect(); torch.cuda.empty_cache()
            return JSONResponse(
                status_code=507,
                content={"error": "CUDA OOM", "detail": str(e),
                         "hint": "Try lower res (e.g. 768×432) or shorter duration (<4 s)."},
            )

    elapsed = round(time.time() - t0, 1)

    # Encode to MP4 in a temp file, read bytes, return.
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        tmp_path = f.name
    try:
        export_to_video(out, tmp_path, fps=body.fps)
        data = open(tmp_path, "rb").read()
    finally:
        try: os.unlink(tmp_path)
        except OSError: pass

    return Response(
        content=data,
        media_type="video/mp4",
        headers={
            "X-Seed":       str(seed),
            "X-Frames":     str(num_frames),
            "X-Elapsed-S":  str(elapsed),
            "X-Width":      str(w),
            "X-Height":     str(h),
        },
    )

print("✅ FastAPI app defined.  Endpoints: GET /health, POST /generate")


In [ ]:
# ==========================================================================
#  CELL 6 — TINY sanity check (1 s × 9 frames, minimum possible).
#  If this OOMs, your GPU has <14 GB usable — lower width/height to 768×432.
#  Skip this if you are in a hurry; the API server works without it.
# ==========================================================================
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("Running 1 s × 9 frame sanity check (tiny workload) ...")
tiny = GenBody(
    prompt="a photorealistic city street at dusk, people walking, cinematic live-action footage",
    width=768, height=432, duration=1.0, fps=9,
    num_inference_steps=8, guidance_scale=5.0, seed=1,
)
try:
    r = generate(tiny)
    if hasattr(r, 'status_code') and r.status_code == 507:
        print("⚠️  Sanity OOM.  Server still works — just use smaller prompts from your laptop.")
    else:
        print(f"✅ Sanity OK  —  MP4 size: {len(r.body):,} bytes,"
              f" seed={r.headers.get('X-Seed')},"
              f" elapsed={r.headers.get('X-Elapsed-S')} s")
except Exception as e:
    print("⚠️  Sanity raised", type(e).__name__ + ":", str(e)[:200])
    print("   (Server is still usable — this was just a warm-up.)")
finally:
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ==========================================================================
#  CELL 7 — Start the HTTP server + print PUBLIC URL.
#  Copy the printed URL into your laptop's .env as:
#      COLAB_BASE_URL=https://xxxx-xxxx-xxxx-xxxx-xxxx.trycloudflare.com
#      (or whatever URL is printed, no trailing slash)
# ==========================================================================
import os, threading, time
import uvicorn, nest_asyncio

nest_asyncio.apply()

PORT = 8000

# lifespan="off" prevents uvicorn from running FastAPI lifespan events in
# the worker thread — no conflict with nest_asyncio's event loop.
config = uvicorn.Config(
    app,
    host="0.0.0.0", port=PORT,
    log_level="warning",
    lifespan="off",
    access_log=False,
)
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# Wait briefly for the socket to bind before trying the proxy.
time.sleep(2.0)

# --------------------------------------------------------------------------
# Try the official Colab proxy first.  This works 95 % of the time.
# If it fails (cookies blocked, permissions, etc.) fall back to LocalTunnel
# then Cloudflared — at least one of the three will print a valid URL.
# --------------------------------------------------------------------------
PUBLIC_URL = None
URL_METHOD = None
FAIL_LOG = []

# 1. Colab kernel proxy (native, no install, no token)
try:
    from google.colab.output import eval_js
    candidate = eval_js(f"google.colab.kernel.proxyPort({PORT})").strip("/")
    if candidate and "http" in candidate and "undefined" not in candidate.lower():
        PUBLIC_URL, URL_METHOD = candidate, "colab-proxy"
    else:
        FAIL_LOG.append(("colab-proxy", f"eval_js returned {candidate!r} (cookies blocked?)"))
except Exception as e:
    FAIL_LOG.append(("colab-proxy", str(e)))

# 2. LocalTunnel (fallback, no token, single npm install)
if PUBLIC_URL is None:
    try:
        import subprocess, json, urllib.request
        subprocess.check_call(["npm", "install", "-g", "localtunnel"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        lt_proc = subprocess.Popen(["lt", "--port", str(PORT)],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True)
        deadline = time.time() + 15
        line = ""
        while time.time() < deadline and PUBLIC_URL is None:
            ch = lt_proc.stdout.read(1)
            if not ch:
                time.sleep(0.2); continue
            if ch == "\n":
                if "your url is:" in line.lower():
                    candidate = line.split(":", 1)[1].strip()
                    if candidate.startswith("http"):
                        PUBLIC_URL, URL_METHOD = candidate, "localtunnel"
                        break
                line = ""
            else:
                line += ch
        if PUBLIC_URL is None:
            lt_proc.terminate()
            FAIL_LOG.append(("localtunnel", "no URL within 15s"))
    except Exception as e:
        FAIL_LOG.append(("localtunnel", str(e)))

# 3. Cloudflared (last fallback — very reliable but binary download)
if PUBLIC_URL is None:
    try:
        import subprocess, shutil
        # Download a tiny static cloudflared binary into /tmp (~45 MB)
        if not shutil.which("cloudflared"):
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            subprocess.check_call(["wget", "-q", "-O", "/tmp/cloudflared", url])
            os.chmod("/tmp/cloudflared", 0o755)
        bin_cf = shutil.which("cloudflared") or "/tmp/cloudflared"
        logfile = open("/tmp/cf.log", "w")
        cf_proc = subprocess.Popen(
            [bin_cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
            stdout=logfile, stderr=logfile,
        )
        deadline = time.time() + 20
        while time.time() < deadline and PUBLIC_URL is None:
            time.sleep(1.0)
            try:
                txt = open("/tmp/cf.log").read()
                for token in txt.split():
                    if token.startswith("https://") and ".trycloudflare.com" in token:
                        PUBLIC_URL, URL_METHOD = token.rstrip("/"), "cloudflared"
                        break
            except Exception:
                pass
        if PUBLIC_URL is None:
            cf_proc.terminate()
            FAIL_LOG.append(("cloudflared", "no URL within 20s"))
    except Exception as e:
        FAIL_LOG.append(("cloudflared", str(e)))

print()
print("=" * 72)
if PUBLIC_URL:
    print(f"✅  TUNNEL READY via {URL_METHOD}")
    print("\n   🔗 PUBLIC API URL — paste into your laptop's .env as COLAB_BASE_URL:")
    print()
    print("   ", PUBLIC_URL)
    print()
    print("   💡 Keep this browser tab open. If Colab disconnects, re-run this")
    print("      cell and update COLAB_BASE_URL to the new URL.")
    print()
    print("   Health check:", PUBLIC_URL + "/health")
else:
    print("🛑  All 3 tunnel methods FAILED.  Details:")
    for m, err in FAIL_LOG:
        print(f"   - {m:14s}: {err}")
    print()
    print("   Manual fix: reload the Colab page, make sure third-party cookies are")
    print("   allowed for colab.googleusercontent.com, and re-run cell 7.")
print("=" * 72)


---

## Optional: quick test from inside Colab (skip if in a hurry)

Once you have the PUBLIC_URL above, you can do an end-to-end test before leaving
this tab:

```python
import requests
r = requests.post(f"{PUBLIC_URL}/generate", json={
    "prompt": "photorealistic zombies walking down an abandoned city street, horror live-action footage, night time, fog",
    "duration": 5.0,
    "width": 832, "height": 480,
    "fps": 16,
    "num_inference_steps": 25,
    "guidance_scale": 5.0,
}, timeout=1200)
print("status:", r.status_code)
if r.status_code == 200:
    open("/content/zombies_test.mp4", "wb").write(r.content)
    print("wrote /content/zombies_test.mp4 — click Files to view")
else:
    print(r.text[:500])
```

Then on your laptop add `COLAB_BASE_URL=<that URL>` to `.env` and run:

```bash
cd your-project
python3 -m src.main gen \
  --backend colab \
  --style realistic \
  --width 832 --height 480 \
  --duration 5 \
  -o outputs/zombies_from_colab.mp4 \
  "photorealistic found footage of a zombie horde shambling down an abandoned downtown city street at dusk, rotting flesh, tattered bloodstained clothes, lifeless milky eyes, reaching arms lunging toward camera, abandoned police cars, shattered storefronts, thick fog, flickering sodium lamps, shaky handheld camcorder, low angle tracking shot, film grain, shot on Sony FX3, 35mm lens, 24fps cinematic"
```
